# 06. ChromaDB 적재·검색 품질 평가

앞 단계가 만든 청크 JSON과 임베딩 NPY를 다시 계산하지 않고 ChromaDB에 적재합니다. 기존 팀 DB를 보호하기 위해 `ㅋㅌㅊ/output/chroma_db`의 전용 컬렉션만 사용합니다. 검색 결과는 같은 문서의 반복 노출을 억제하고 앞뒤 청크를 연결해 잘린 문맥을 보완합니다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import re
import shutil
import tempfile
import warnings

os.environ.setdefault('HF_HUB_DISABLE_IMPLICIT_TOKEN_WARNING', '1')
warnings.filterwarnings('ignore', message='IProgress not found.*')

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import numpy as np
import pandas as pd
from IPython.display import display

def find_data_dir():
    for candidate in [Path.cwd(), Path.cwd() / 'ㅋㅌㅊ', Path.cwd().parent, Path.cwd().parent / 'ㅋㅌㅊ']:
        resolved = candidate.resolve()
        if (resolved / 'output/RAG/maple_inven_tips_embeddings_manifest.json').is_file():
            return resolved
    raise FileNotFoundError('01~05 노트북을 먼저 순서대로 실행하세요.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
CHUNKS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_documents_chunked.json'
EMBEDDINGS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_embeddings.npy'
MANIFEST_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_embeddings_manifest.json'
SEARCH_REPORT_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_search_report.json'
CHROMA_DIR = OUTPUT_ROOT / 'chroma_db'
COLLECTION_NAME = 'maplestory_inven_tips'
UPSERT_BATCH_SIZE = 500
CANDIDATE_K = 20
TOP_K = 5
LOW_CONFIDENCE_DISTANCE = 0.65

print('전용 Chroma 경로:', CHROMA_DIR)
print('컬렉션:', COLLECTION_NAME)

전용 Chroma 경로: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\ㅋㅌㅊ\output\chroma_db
컬렉션: maplestory_inven_tips


In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
vectors = np.load(EMBEDDINGS_PATH, allow_pickle=False)
chunk_ids = [chunk['id'] for chunk in chunks]
norms = np.linalg.norm(vectors, axis=1)

assert len(chunks) == vectors.shape[0] == manifest['embedding_count']
assert vectors.ndim == 2 and vectors.shape[1] == manifest['embedding_dimension']
assert vectors.dtype == np.float32 and np.isfinite(vectors).all()
assert np.allclose(norms, 1.0, atol=1e-5)
assert chunk_ids == manifest['chunk_ids'] and len(chunk_ids) == len(set(chunk_ids))
assert sha256_file(CHUNKS_PATH) == manifest['chunks_sha256']
assert sha256_file(EMBEDDINGS_PATH) == manifest['embeddings_sha256']

display({
    '청크 수': len(chunks), '임베딩 shape': vectors.shape,
    'dtype': str(vectors.dtype), '정규화': manifest['normalized'],
    'JSON checksum': '통과', 'NPY checksum': '통과',
})

{'청크 수': 5506,
 '임베딩 shape': (5506, 768),
 'dtype': 'float32',
 '정규화': True,
 'JSON checksum': '통과',
 'NPY checksum': '통과'}

In [3]:
def sanitize_metadata(metadata):
    sanitized = {}
    for key, value in metadata.items():
        if value is None:
            continue
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, (str, int, float, bool)):
            sanitized[str(key)] = value
        else:
            sanitized[str(key)] = json.dumps(value, ensure_ascii=False)
    return sanitized

query_embedding_function = SentenceTransformerEmbeddingFunction(
    model_name=manifest['model_name'], normalize_embeddings=True,
)

# 같은 커널에서 이 셀을 다시 실행할 때 이전 DB 핸들을 먼저 닫는다.
previous_client = globals().pop('client', None)
globals().pop('collection', None)
if previous_client is not None:
    previous_client.close()
    del previous_client
    gc.collect()

# 이 노트북 전용 DB만 초기화해 반복 실행 시 고아 HNSW 인덱스가 남지 않게 한다.
expected_chroma_dir = (OUTPUT_ROOT / 'chroma_db').resolve()
resolved_chroma_dir = CHROMA_DIR.resolve()
if resolved_chroma_dir != expected_chroma_dir or resolved_chroma_dir.parent != OUTPUT_ROOT.resolve():
    raise RuntimeError(f'안전하지 않은 Chroma 초기화 경로: {resolved_chroma_dir}')
if resolved_chroma_dir.exists():
    shutil.rmtree(resolved_chroma_dir)
resolved_chroma_dir.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(resolved_chroma_dir))

collection = client.create_collection(
    name=COLLECTION_NAME, embedding_function=query_embedding_function,
    metadata={'hnsw:space': 'cosine', 'description': '메이플 인벤 팁과 노하우'},
)
for start in range(0, len(chunks), UPSERT_BATCH_SIZE):
    end = min(start + UPSERT_BATCH_SIZE, len(chunks))
    batch = chunks[start:end]
    collection.add(
        ids=[chunk['id'] for chunk in batch],
        embeddings=vectors[start:end].tolist(),
        documents=[chunk['page_content'] for chunk in batch],
        metadatas=[sanitize_metadata(chunk['metadata']) for chunk in batch],
    )

stored_count = collection.count()
assert stored_count == len(chunks)
display({
    '저장 경로': str(CHROMA_DIR), '컬렉션': COLLECTION_NAME,
    '적재 청크': stored_count, '거리 함수': 'cosine',
    '기존 팀 DB 변경': False,
})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7370.25it/s]

{'저장 경로': 'C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\output\\chroma_db',
 '컬렉션': 'maplestory_inven_tips',
 '적재 청크': 5506,
 '거리 함수': 'cosine',
 '기존 팀 DB 변경': False}

In [4]:
print('질의 임베딩 모델:', manifest['model_name'])
print('벡터 차원:', manifest['embedding_dimension'])

chunk_by_document_index = {
    (chunk['metadata']['document_id'], int(chunk['metadata']['chunk_index'])): chunk
    for chunk in chunks
}

def merge_with_overlap(left, right, minimum_overlap=20):
    if not left:
        return right
    if not right or right in left:
        return left
    maximum = min(len(left), len(right), 500)
    for size in range(maximum, minimum_overlap - 1, -1):
        if left[-size:] == right[:size]:
            return left + right[size:]
    return left + '\n' + right

def build_context_window(metadata, radius=1):
    document_id = metadata['document_id']
    center_index = int(metadata['chunk_index'])
    neighbor_chunks = []
    for index in range(max(0, center_index - radius), center_index + radius + 1):
        chunk = chunk_by_document_index.get((document_id, index))
        if chunk is not None:
            neighbor_chunks.append(chunk)
    context = ''
    for chunk in neighbor_chunks:
        context = merge_with_overlap(context, chunk['page_content'])
    return context, [chunk['id'] for chunk in neighbor_chunks]

def raw_search(query_vector, candidate_k):
    response = collection.query(
        query_embeddings=query_vector.tolist(),
        n_results=min(candidate_k, collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )
    return [
        {'chunk_id': chunk_id, 'document': document, 'metadata': metadata, 'distance': float(distance)}
        for chunk_id, document, metadata, distance in zip(
            response['ids'][0], response['documents'][0],
            response['metadatas'][0], response['distances'][0],
        )
    ]

def retrieve(query, top_k=TOP_K, candidate_k=CANDIDATE_K, context_radius=1):
    query_vector = np.asarray(query_embedding_function([query]), dtype=np.float32)
    requested_candidates = min(max(candidate_k, top_k), collection.count())
    while True:
        candidates = raw_search(query_vector, candidate_k=requested_candidates)
        unique_document_count = len({item['metadata']['document_id'] for item in candidates})
        if unique_document_count >= top_k or requested_candidates >= collection.count():
            break
        requested_candidates = min(requested_candidates * 2, collection.count())
    selected = []
    seen_documents = set()
    for candidate in candidates:
        document_id = candidate['metadata']['document_id']
        if document_id in seen_documents:
            continue
        seen_documents.add(document_id)
        context, neighbor_ids = build_context_window(candidate['metadata'], radius=context_radius)
        selected.append({
            'rank': len(selected) + 1,
            'chunk_id': candidate['chunk_id'],
            'document_id': document_id,
            'article_id': str(candidate['metadata'].get('article_id', '')),
            'title': candidate['metadata'].get('name', '제목 없음'),
            'category': candidate['metadata'].get('section_title', '기타'),
            'url': candidate['metadata'].get('url', ''),
            'distance': candidate['distance'],
            'similarity': 1.0 - candidate['distance'],
            'neighbor_ids': neighbor_ids,
            'candidate_pool_size': requested_candidates,
            'context': context,
        })
        if len(selected) >= top_k:
            break
    return selected

질의 임베딩 모델: jhgan/ko-sroberta-multitask
벡터 차원: 768


## 검색 결과 직접 확인

`QUERY`를 바꿔 다시 실행하면 Top-5와 앞뒤 청크가 결합된 문맥을 확인할 수 있습니다.

In [5]:
QUERY = '쿨뚝과 메르세데스 유니온 효과로 쿨타임이 얼마나 감소하나요?'
search_results = retrieve(QUERY)
display(pd.DataFrame([{
    'rank': item['rank'], 'distance': round(item['distance'], 4),
    'article_id': item['article_id'], 'category': item['category'],
    'title': item['title'], 'neighbor_ids': item['neighbor_ids'],
    'candidate_pool_size': item['candidate_pool_size'],
} for item in search_results]))
print('\n[Top-1 확장 문맥]')
print(search_results[0]['context'][:2000] if search_results else '검색 결과 없음')

,rank,distance,article_id,category,title,neighbor_ids,candidate_pool_size
0,1,0.3160,48082,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,"[inven_tip_48082_5, inven_tip_48082_6, inven_t...",40
1,2,0.4225,43854,기타,"◆ 2분 주기와 쿨뚝메타, 시드링에 대하여 ◆","[inven_tip_43854_12, inven_tip_43854_13, inven...",40
2,3,0.4698,43410,기타,[3/20 적용]개편&바뀌는 것들 요약 + 캐시 / 신규 이벤트,"[inven_tip_43410_24, inven_tip_43410_25, inven...",40
3,4,0.4806,45693,기타,"(스압, 데이터주의) 전직업 시너지 효과표 2025ver","[inven_tip_45693_21, inven_tip_45693_22, inven...",40
4,5,0.5087,42665,기타,[NEXT 2차]패치후 메할일&변경사항요약+@ : 1/16~,"[inven_tip_42665_32, inven_tip_42665_33, inven...",40



[Top-1 확장 문맥]
또한 설명에 나와있듯이 잠재능력보다 우선으로 적용되기 때문에 쿨타임이 서로 다른 스킬들에도 모두 똑같은 효율을 가집니다.
스킬의 쿨타임이 감소했을 때 얻는 이점은 아래에서 따로 설명하겠습니다.
3. 쿨뚝? 2초뚝? 5초뚝? 이게 뭘까
메이플에는 스킬의 쿨타임을 감소시킬 수 있는 방법이 총 2가지가 있는데, 하나는 방금 말씀드렸던 메르세데스의 유니온(챌린저스 버프)이며, 나머지 하나는 모자에 붙는 레전드리 잠재능력을 이용하는 것입니다.
이 쿨타임 감소 잠재능력이 부여된 모자를 통틀어서 '쿨뚝' 이라고 부르며,
이 단어 앞에 붙은 숫자는 해당 장비의 잠재능력에 부여된 모든 쿨타임 감소 수치를 합한 값입니다.
2초뚝이라고 하면 쿨타임 감소 옵션이 1초x2줄 or 2초x1줄 붙어있는 모자라는 뜻입니다.
(+ '2초뚝', '2초쿨뚝', '2초' 모두 같은 의미입니다.)


## 대표 질문 검색 평가

예상 게시글의 Top-5 적중 여부(Hit@5), 기대 문서 중 실제로 찾은 비율(Recall@5), MRR을 확인합니다. 미적중·부분 재현·낮은 신뢰도 사례는 별도로 출력합니다.

In [6]:
EVALUATION_CASES = [
    {'query': '쿨뚝과 메르세데스 유니온 효과로 쿨타임이 얼마나 감소하나요?', 'expected': ['48082']},
    {'query': '메이플M 렌을 무과금으로 250까지 키우는 방법', 'expected': ['48066', '46482']},
    {'query': '울티마 스쿼드 장비와 잠재 옵션 정보', 'expected': ['48012', '47984']},
    {'query': '세르니움에서 야누스 30레벨 제자리 사냥터 추천', 'expected': ['47636']},
    {'query': '스타포스 파괴 후 확정 복구를 어떻게 사용하나요?', 'expected': ['47118']},
    {'query': '유니온 1만 이상 자동배치 미세 팁', 'expected': ['47020']},
    {'query': '루시드 보스 스킬 딜 사이클 설명', 'expected': ['47189']},
    {'query': '렌 보스전 극딜에서 평딜로 넘어가는 방법', 'expected': ['44718', '44685']},
]

evaluation_rows = []
failure_details = []
reciprocal_ranks = []
for case in EVALUATION_CASES:
    results = retrieve(case['query'])
    retrieved_ids = [result['article_id'] for result in results]
    expected_ids = set(case['expected'])
    retrieved_relevant_ids = expected_ids.intersection(retrieved_ids)
    hit_rank = next((index + 1 for index, article_id in enumerate(retrieved_ids) if article_id in expected_ids), None)
    hit_at_5 = hit_rank is not None
    case_recall_at_5 = len(retrieved_relevant_ids) / len(expected_ids)
    reciprocal_ranks.append(0.0 if hit_rank is None else 1.0 / hit_rank)
    best_distance = results[0]['distance'] if results else None
    low_confidence = best_distance is None or best_distance > LOW_CONFIDENCE_DISTANCE
    candidate_pool_size = results[0]['candidate_pool_size'] if results else 0
    evaluation_rows.append({
        'query': case['query'], 'expected': ', '.join(case['expected']),
        'hit_rank': hit_rank, 'hit@5': hit_at_5, 'recall@5': case_recall_at_5,
        'best_distance': None if best_distance is None else round(best_distance, 4),
        'low_confidence': low_confidence,
        'candidate_pool_size': candidate_pool_size,
        'top5_article_ids': retrieved_ids,
    })
    if not hit_at_5 or case_recall_at_5 < 1.0 or low_confidence:
        failure_details.append({
            'query': case['query'], 'expected': case['expected'],
            'hit_rank': hit_rank, 'hit@5': hit_at_5, 'recall@5': case_recall_at_5,
            'missing_expected_ids': sorted(expected_ids - retrieved_relevant_ids),
            'results': [{
                'rank': item['rank'], 'article_id': item['article_id'],
                'title': item['title'], 'distance': round(item['distance'], 4),
            } for item in results],
        })

hit_at_5 = sum(row['hit@5'] for row in evaluation_rows) / len(evaluation_rows)
recall_at_5 = sum(row['recall@5'] for row in evaluation_rows) / len(evaluation_rows)
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
search_report = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'persist_directory': str(CHROMA_DIR), 'collection_name': COLLECTION_NAME,
    'collection_count': collection.count(), 'candidate_k': CANDIDATE_K, 'top_k': TOP_K,
    'document_deduplication': True, 'adaptive_candidate_expansion': True, 'context_radius': 1,
    'evaluation_case_count': len(EVALUATION_CASES),
    'hit_at_5': hit_at_5, 'recall_at_5': recall_at_5, 'mrr': mrr,
    'failure_count': sum(not row['hit@5'] for row in evaluation_rows),
    'partial_recall_count': sum(row['recall@5'] < 1.0 for row in evaluation_rows),
    'low_confidence_count': sum(row['low_confidence'] for row in evaluation_rows),
    'evaluation': evaluation_rows, 'failure_details': failure_details,
}
atomic_write_json(SEARCH_REPORT_PATH, search_report)

display({'Hit@5': hit_at_5, 'Recall@5': recall_at_5, 'MRR': mrr, '실패 질의': search_report['failure_count'], '부분 재현': search_report['partial_recall_count'], '낮은 신뢰도': search_report['low_confidence_count']})
display(pd.DataFrame(evaluation_rows))
print('검색 미적중·부분 재현·낮은 신뢰도 상세')
display(pd.DataFrame(failure_details))

{'Hit@5': 1.0,
 'Recall@5': 1.0,
 'MRR': 0.9375,
 '실패 질의': 0,
 '부분 재현': 0,
 '낮은 신뢰도': 0}

,query,expected,hit_rank,hit@5,recall@5,best_distance,low_confidence,candidate_pool_size,top5_article_ids
0,쿨뚝과 메르세데스 유니온 효과로 쿨타임이 얼마나 감소하나요?,48082,1,True,1.0,0.3160,False,40,"[48082, 43854, 43410, 45693, 42665]"
1,메이플M 렌을 무과금으로 250까지 키우는 방법,"48066, 46482",2,True,1.0,0.4032,False,40,"[42050, 46482, 48066, 41923, 42181]"
2,울티마 스쿼드 장비와 잠재 옵션 정보,"48012, 47984",1,True,1.0,0.2068,False,80,"[48012, 47971, 47984, 46759, 39782]"
3,세르니움에서 야누스 30레벨 제자리 사냥터 추천,47636,1,True,1.0,0.2401,False,20,"[47636, 47635, 46468, 39985, 47515]"
4,스타포스 파괴 후 확정 복구를 어떻게 사용하나요?,47118,1,True,1.0,0.3675,False,20,"[47118, 43522, 43466, 43410, 43440]"
5,유니온 1만 이상 자동배치 미세 팁,47020,1,True,1.0,0.3770,False,20,"[47020, 42352, 40240, 42598, 44883]"
6,루시드 보스 스킬 딜 사이클 설명,47189,1,True,1.0,0.2917,False,20,"[47189, 47490, 43569, 47805, 43262]"
7,렌 보스전 극딜에서 평딜로 넘어가는 방법,"44718, 44685",1,True,1.0,0.2053,False,20,"[44718, 44685, 47899, 42970, 47515]"


검색 미적중·부분 재현·낮은 신뢰도 상세


""
